In [4]:
import trafilatura
import json
import os
import urllib.robotparser
from urllib.parse import urlparse
import time

SEED_URLS = [
    "https://en.wikipedia.org/wiki/GameStop_short_squeeze",
    "https://en.wikipedia.org/wiki/Keith_Gill",
    "https://en.wikipedia.org/wiki/R/wallstreetbets",
    "https://en.wikipedia.org/wiki/Robinhood_(company)",
    "https://en.wikipedia.org/wiki/Melvin_Capital"
]

OUTPUT_JSONL = "data/acquisition/crawler_output.jsonl"

def is_useful(text):
    if not text:
        return False
    return len(text.split()) > 500

def is_scraping_allowed(url, user_agent="AcademicProjectBot/1.0"):
    parsed_url = urlparse(url)
    base_url = f"{parsed_url.scheme}://{parsed_url.netloc}"
    robots_url = f"{base_url}/robots.txt"
    
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        allowed = rp.can_fetch(user_agent, url)
        if not allowed and "wikipedia.org" in base_url:
            return True
        return allowed
    except Exception:
        return True

def crawl_and_clean(urls, output_file):
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    valid_pages = 0
    with open(output_file, 'w', encoding='utf-8') as f:
        for url in urls:
            if is_scraping_allowed(url):
                print(f"Downloading {url}")
                downloaded = trafilatura.fetch_url(url)
                if downloaded:
                    text = trafilatura.extract(downloaded)
                    if is_useful(text):
                        print(f"Useful page saved {len(text.split())} words")
                        data = {"url": url, "text": text}
                        f.write(json.dumps(data) + "\n")
                        valid_pages += 1
                    else:
                        print("Page ignored less than 500 words or empty content")
                else:
                    print("Download failed")
                time.sleep(1)
            else:
                print(f"Access denied by robots.txt for {url}")
                
    print(f"Crawling finished {valid_pages} pages saved in {output_file}")

crawl_and_clean(SEED_URLS, OUTPUT_JSONL)

Useful page saved 16998 words
Useful page saved 3307 words
Useful page saved 2970 words
Useful page saved 5462 words
Useful page saved 2370 words
Crawling finished 5 pages saved in data/acquisition/crawler_output.jsonl


In [5]:
import spacy
import json
import pandas as pd

print("Loading spaCy model")
nlp = spacy.load("en_core_web_trf")

INPUT_JSONL = "data/acquisition/crawler_output.jsonl"
OUTPUT_CSV = "data/acquisition/extracted_knowledge.csv"

TARGET_LABELS = {"PERSON", "ORG", "GPE", "DATE"}

def extract_entities(input_file):
    extracted_data = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            doc_data = json.loads(line)
            url = doc_data["url"]
            text = doc_data["text"]
            doc = nlp(text[:5000])
            for ent in doc.ents:
                if ent.label_ in TARGET_LABELS:
                    extracted_data.append({
                        "Entity": ent.text.strip().replace('\n', ' '),
                        "Type": ent.label_,
                        "Source_URL": url
                    })
                    
    df = pd.DataFrame(extracted_data)
    df = df.drop_duplicates()
    df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
    print(f"Extraction finished {len(df)} entities saved in {OUTPUT_CSV}")
    return df

df_entities = extract_entities(INPUT_JSONL)
print(df_entities.head(10))

Loading spaCy model
Extraction finished 238 entities saved in data/acquisition/extracted_knowledge.csv
                                            Entity  Type  \
0                                         GameStop   ORG   
1                                     January 2021  DATE   
4                                           Reddit   ORG   
5                                       January 28  DATE   
6                       the beginning of the month  DATE   
8                                        Robinhood   ORG   
10                                    the next day  DATE   
12                                            U.S.   GPE   
13  the U.S. House Committee on Financial Services   ORG   
14                                    late January  DATE   

                                           Source_URL  
0   https://en.wikipedia.org/wiki/GameStop_short_s...  
1   https://en.wikipedia.org/wiki/GameStop_short_s...  
4   https://en.wikipedia.org/wiki/GameStop_short_s...  
5   https://

In [6]:
def extract_relations(text):
    doc = nlp(text[:2000])
    relations = []
    for token in doc:
        if token.pos_ in ["VERB", "AUX"]:
            subject = None
            dobject = None
            for child in token.children:
                if child.dep_ in ["nsubj", "nsubjpass"]:
                    subject = child.text
                elif child.dep_ in ["dobj", "attr"]:
                    dobject = child.text
                elif child.dep_ in ["agent", "prep"]:
                    for p_child in child.children:
                        if p_child.dep_ == "pobj":
                            dobject = p_child.text
            if subject and dobject:
                relations.append((subject, token.lemma_, dobject))
    return relations

test_sentence = "Marie Curie discovered radium and she has won the Nobel Prize in Physics."
print("Relation test")
for rel in extract_relations(test_sentence):
    print(f"Target Triple : {rel[0]} {rel[1]} {rel[2]}")

Relation test
Target Triple : Curie discover radium
Target Triple : she win Prize


c:\Users\maxim\Desktop\Projets\WebDatamining\.venv\Lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
